# Week1_ex5 - Induction Heating of a Cast-Iron Disk

This exercise computes the eddy currents induced in a cast-iron disk by a helical coil carrying 125 A at 500 Hz, modeled as a simplified induction heating setup using the EddyCurrent solver in Maxwell 3D.

In [ ]:
import pandas as pd
import numpy as np
import os
import ansys.aedt.core
import math
import shutil
import time

In [ ]:
## Create Desktop, Project, and Design ##

# creating Desktop object
# write the aedt(ansys electronics desktop) version and whether you are using a student license
DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)

# Option to automatically save at regular intervals
# disabled because the simulation is run on a script basis
DT.disable_autosave()

# solution type
sol_type = "EddyCurrent"

# creating maxwell3D design object
# write solution type and whether you are using a student license
# when a design object is first created, it comes with a new project, and both are given arbitrary names
M3D = ansys.aedt.core.maxwell.Maxwell3d(solution_type=sol_type)

# odesign object, needed to use Ansys-recorded script commands
oDesign = M3D.odesign

In [ ]:
## Set up output directory for results ##

proj_name = "Week1_ex5"

# If ANSYS_PROJECT_DIR is set on this machine, save there.
# Otherwise, fall back to the notebook's own working directory,
# so this stays portable for anyone else running the notebook as-is.
base_dir = os.environ.get("ANSYS_PROJECT_DIR", os.getcwd())
dir = os.path.join(base_dir, proj_name)
print(dir)

# Create directory if it doesn't exist yet (safe to re-run; won't error if already there)
os.makedirs(dir, exist_ok=True)

desi_name = "Week1_ex5"


In [ ]:
## Save project and apply design name ##

proj = M3D.oproject

# Save project with the target file name (directory created above)
proj.SaveAs(f"{dir}\\{proj_name}.aedt", True)

# Rename design to match the target design name
M3D.rename_design(desi_name, save=False)

# Save again so the design-name change is committed to disk
M3D.save_project()

In [ ]:
## Define helper functions ##

def find_min_positive_index(numbers) :      # Given a list, returns the index of the positive value with the smallest absolute value
    # Keep only positive numbers
    positive_numbers = [(index, num) for index, num in enumerate(numbers) if num > 0]
    
    if not positive_numbers:
        # Return None if there are no positive numbers
        return None
    
    # Find the minimum by absolute value
    min_index, _ = min(positive_numbers, key=lambda x: abs(x[1]))
    return min_index

In [ ]:
## Create geometry ##

# Change units

M3D.modeler.model_units = "cm"


# Draw the coil

# NOTE: helix dimensions are pulled out as variables (rather than hardcoded inside the
# UDP call below) because the lead-box positions further down depend on where the helix
# physically ends. Turns reduced from 8 to 4 here to keep the total mesh element count
# under the Student license's 3,000-element cap; SegmentsPerTurn similarly reduced.
helix_start_radius = 15      # cm
helix_radius_change = 3.1    # cm per turn
helix_turns = 4
helix_segments_per_turn = 12

oEditor = oDesign.SetActiveEditor("3D Modeler")		# copied from AEDT's script recording feature
oEditor.CreateUserDefinedPart(
	[
		"NAME:UserDefinedPrimitiveParameters",
		"DllName:="		, "SegmentedHelix/PolygonHelix.dll",
		"Version:="		, "1.0",
		"NoOfParameters:="	, 8,
		"PerformIDTranslation:=", False,
		"Library:="		, "syslib",
		[
			"NAME:ParamVector",
			[
				"NAME:Pair",
				"Name:="		, "PolygonSegments",
				"Value:="		, "4"
			],
			[
				"NAME:Pair",
				"Name:="		, "PolygonRadius",
				"Value:="		, "1.5cm"
			],
			[
				"NAME:Pair",
				"Name:="		, "StartHelixRadius",
				"Value:="		, f"{helix_start_radius}cm"
			],
			[
				"NAME:Pair",
				"Name:="		, "RadiusChange",
				"Value:="		, f"{helix_radius_change}cm"
			],
			[
				"NAME:Pair",
				"Name:="		, "Pitch",
				"Value:="		, "0cm"
			],
			[
				"NAME:Pair",
				"Name:="		, "Turns",
				"Value:="		, f"{helix_turns}"
			],
			[
				"NAME:Pair",
				"Name:="		, "SegmentsPerTurn",
				"Value:="		, f"{helix_segments_per_turn}"
			],
			[
				"NAME:Pair",
				"Name:="		, "RightHanded",
				"Value:="		, "1"
			]
		]
	], 
	[
		"NAME:Attributes",
		"Name:="		, "PolygonHelix1",
		"Flags:="		, "",
		"Color:="		, "(143 175 143)",
		"Transparency:="	, 0,
		"PartCoordinateSystem:=", "Global",
		"UDMId:="		, "",
		"MaterialValue:="	, "\"vacuum\"",
		"SurfaceMaterialValue:=", "\"\"",
		"SolveInside:="		, True,
		"ShellElement:="	, False,
		"ShellElementThickness:=", "0cm",
		"ReferenceTemperature:=", "20cel",
		"IsMaterialEditable:="	, True,
		"IsSurfaceMaterialEditable:=", True,
		"UseMaterialAppearance:=", False,
		"IsLightweight:="	, False
	])

coil = M3D.modeler.object_list[-1]
M3D.assign_material(assignment=coil, material="copper")

# NOTE: box1/box2 are the coil's lead-in/lead-out connectors. box1 sits near the
# helix's starting radius (fixed — doesn't depend on Turns). box2 must sit near the
# helix's *ending* radius, which does depend on Turns, so its x-position is computed
# from the same variables used above rather than hardcoded — this is what broke
# ("Current leak to the air") the first time Turns was changed without updating this.
lead_margin = 0.7    # cm of overlap between the lead box and the coil end, matching the original design
helix_end_radius = helix_start_radius + helix_turns * helix_radius_change
box2_x = helix_end_radius + lead_margin

origin = [14, 0, -2]
sizes = [2, 2, -2]
box1 = M3D.modeler.create_box(origin=origin, sizes=sizes, name=None, material=None)

origin = [box2_x, 0, -2]
sizes = [-2, -2, -2]
box2 = M3D.modeler.create_box(origin=origin, sizes=sizes, name=None, material=None)

face1 = box1.faces[0]	# find the face with the largest center x-coordinate
for f in box1.faces :
    if ( face1.center[0] < f.center[0] ) :
        face1 = f

face2 = box2.faces[0]	# find the face with the smallest center x-coordinate
for f in box2.faces :
    if ( face2.center[0] > f.center[0] ) :
        face2 = f

face1 = M3D.modeler.create_object_from_face(assignment=face1, non_model=False)
face2 = M3D.modeler.create_object_from_face(assignment=face2, non_model=False)
M3D.modeler.connect(assignment=[face1, face2])
connector = M3D.modeler.object_list[-1]

vector = [0, 0, 1]
M3D.modeler.duplicate_along_line(assignment=box1, vector=vector, clones=2, attach=False, is_3d_comp=False, duplicate_assignment=True)
tmp = M3D.modeler.object_list[-1]
M3D.modeler.unite(assignment=[box1, tmp], purge=False, keep_originals=False)

vector = [0, 0, 1]
M3D.modeler.duplicate_along_line(assignment=box2, vector=vector, clones=2, attach=False, is_3d_comp=False, duplicate_assignment=True)
tmp = M3D.modeler.object_list[-1]
M3D.modeler.unite(assignment=[box2, tmp], purge=False, keep_originals=False)

M3D.modeler.unite(assignment=[coil, box1, box2, connector], purge=False, keep_originals=False)


# Draw the disk
# NOTE: num_sides reduced from 36 to 12 (same reasoning as the region below)
# to help keep the total mesh element count under the Student license's 3,000-element cap.

center = [0, 0, 1.5]
origin = [41, 0, 1.5]
height = 1
disk = M3D.modeler.create_polyhedron(orientation=None, center=center, origin=origin, height=height, num_sides=12, name="Disk", material="cast_iron")


In [ ]:
## Assign current excitation ##

M3D.modeler.section(assignment=coil, plane="YZ", create_new=True, section_cross_object=False)

coil_terminal = M3D.modeler.sheet_objects[-1]
M3D.modeler.separate_bodies(assignment=coil_terminal, create_group=False)

sheets_y = []
for s in M3D.modeler.sheet_objects :    # find the center y-coordinate of each sheet object
    sheets_y.append(s.faces[0].center[1])

terminal_inx = find_min_positive_index(sheets_y)

coil_terminal = M3D.modeler.sheet_objects[terminal_inx]

tmp = 0
for s in M3D.modeler.sheet_objects :    # delete every sheet except the one we want to keep
    if tmp != terminal_inx :
        M3D.modeler.delete(assignment=s)
    tmp += 1

M3D.assign_current(assignment=coil_terminal, amplitude="125A", phase='0deg', solid=True, swap_direction=False, name="I_Coil")

In [ ]:
## Set up mesh layers for skin effect ##

tmp = disk.bottom_face_z
M3D.modeler.create_object_from_face(assignment=tmp, non_model=False)
disk_face = M3D.modeler.sheet_objects[-1]

vector = [0, 0, 0.125]
disk_face.move(vector)
M3D.modeler.duplicate_along_line(assignment=disk_face, vector=vector, clones=2, attach=False, is_3d_comp=False, duplicate_assignment=True)


In [ ]:
## Set up the simulation region ##

# NOTE: region size and facet count (num_sides) reduced from the original exercise
# (radius 150 -> 100, height 100 -> 60, num_sides 36 -> 12) to help keep the total
# mesh element count under the Student license's 3,000-element cap.
center = [0, 0, -30]
origin = [100, 0, -30]
height = 60
region = M3D.modeler.create_polyhedron(orientation=None, center=center, origin=origin, height=height, 
                                       num_sides=12, name="Region", material=None,)

In [ ]:
## Configure eddy effects ##

M3D.eddy_effects_on(assignment=coil.name, enable_eddy_effects=False,  enable_displacement_current=False)
M3D.eddy_effects_on(assignment=disk.name, enable_eddy_effects=True,  enable_displacement_current=False)


In [ ]:
## Configure analysis setup ##

# setup object
# NOTE: setup_type must be passed explicitly here (same fix as Week1_ex4).
# In AEDT 2025.2, the EddyCurrent solver reports its internal solution type as "AC Magnetic",
# but pyaedt 0.15.3's automatic lookup only recognizes "EddyCurrent" as a key,
# so letting it auto-detect raises a KeyError and create_setup() silently returns False.
my_setup = M3D.create_setup(name="Setup1", setup_type="EddyCurrent")

# check the Analysis setup properties for the current solution type
# stored as a dictionary in the setup object's props attribute
display(my_setup.props)

In [ ]:
# modify the desired fields from the props displayed above

# NOTE: MaximumPasses=15 alone wasn't enough — the solver kept refining past the
# license's mesh cap (74,244 elements, still failed; known bounds so far: ~32,330
# succeeds, ~64,484 and up fails). Lowering PercentRefinement means each pass adds
# mesh more slowly, which may allow convergence to happen before hitting the cap.
my_setup.props['MaximumPasses'] = 15
my_setup.props['PercentError'] = 2
my_setup.props['PercentRefinement'] = 15
my_setup.props['Frequency'] = "500Hz"


In [ ]:
## Run analysis ##

my_setup.analyze()

In [ ]:
# Save final results

M3D.save_project()